# Week 3 — Residual-Based Spoofing Detection

You are given pseudorange residuals for multiple satellites over time.
A spoofing event begins partway through the dataset.

Your goal is to build a simple detector.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

df = pd.read_csv("../datasets/week3_residual_detection.csv")
df.head()

## Task 1: Aggregate residuals across satellites

For each time step, compute a summary statistic such as:
- mean absolute residual
- RMS residual
- maximum absolute residual

In [ ]:
agg = df.groupby("t_s").agg(
    mean_abs_residual_m=("residual_m", lambda s: np.mean(np.abs(s))),
    rms_residual_m=("residual_m", lambda s: np.sqrt(np.mean(np.square(s)))),
    truth_spoof=("spoof_active", "max")
).reset_index()

agg.head()

In [ ]:
fig, ax = plt.subplots(figsize=(10,5))
ax.plot(agg["t_s"], agg["rms_residual_m"], label="RMS residual")
ax.set_xlabel("Time [s]")
ax.set_ylabel("Residual metric [m]")
ax.set_title("Residual metric over time")
ax.legend()
plt.show()

## Task 2: Build a threshold detector

Pick a threshold on RMS residual and create a detection flag.

In [ ]:
threshold = 8.0  # TODO: tune this
agg["detected"] = (agg["rms_residual_m"] > threshold).astype(int)

fig, ax = plt.subplots(figsize=(10,4))
ax.plot(agg["t_s"], agg["truth_spoof"], label="truth")
ax.plot(agg["t_s"], agg["detected"], label="detected", alpha=0.8)
ax.set_xlabel("Time [s]")
ax.set_ylabel("Binary flag")
ax.legend()
plt.show()

## Task 3: Compute performance metrics

Report:
- true positives
- false positives
- false negatives
- detection delay

In [ ]:
tp = int(((agg["detected"] == 1) & (agg["truth_spoof"] == 1)).sum())
fp = int(((agg["detected"] == 1) & (agg["truth_spoof"] == 0)).sum())
fn = int(((agg["detected"] == 0) & (agg["truth_spoof"] == 1)).sum())

first_truth = agg.loc[agg["truth_spoof"] == 1, "t_s"].min()
first_det = agg.loc[agg["detected"] == 1, "t_s"].min()
delay = None if pd.isna(first_det) else first_det - first_truth

{"tp": tp, "fp": fp, "fn": fn, "detection_delay_s": delay}

## Stretch task
Replace the hand-tuned threshold with a threshold based on the pre-attack statistics, for example:
\[
\text{threshold} = \mu_{\text{pre}} + 3\sigma_{\text{pre}}
\]